<a href="https://colab.research.google.com/github/toecm/iedi-mas/blob/December-2025/IEDI_M%C2%B2_Conditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# -*- coding: utf-8 -*-
"""IEDI-M² Final Integrated Version
(Dynamic Model Switching + Rich Context + Lab Persona and Auto-Switching: Pro -> Flash on Exhaustion)
"""

# --- INSTALL DEPENDENCIES ---
!pip install -q openai-whisper rapidfuzz pandas gradio datasets transformers torchaudio torch librosa pydub ffmpeg-python jiwer google-generativeai python-dotenv requests yt-dlp soundfile

import os
import glob
import torch
import whisper
import pandas as pd
import requests
import tempfile
import yt_dlp
import random
import soundfile as sf
from pydub import AudioSegment
from rapidfuzz import process, fuzz
import google.generativeai as genai
from google.api_core import exceptions as google_exceptions
from datasets import load_dataset, Audio
import gradio as gr
from dotenv import load_dotenv
from threading import Lock
from huggingface_hub import HfApi, hf_hub_download, upload_file
import json
import re
import traceback
import shutil

# --- CONFIGURATION ---
HF_REPO_ID = "toecm/IEDID"

# Load Environment Variables
load_dotenv()

# Try Loading Keys from Colab Secrets (Preferred)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') or os.getenv("HF_TOKEN")
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') or os.getenv("GOOGLE_API_KEY")
    os.environ["PINATA_JWT"] = userdata.get('PINATA_JWT') or os.getenv("PINATA_JWT")
except (ImportError, Exception):
    pass

# Final Key Assignment
HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINATA_JWT = os.getenv("PINATA_JWT")
DATASET_DIR = "/content/iuuy_datasets"
os.makedirs(DATASET_DIR, exist_ok=True)

# --- DYNAMIC MODEL MANAGER (Auto-Switch Logic) ---
class GeminiManager:
    """
    Manages Google Gemini models with automatic fallback.
    Attempt 1: Gemini 1.5 Pro
    If (ResourceExhausted):
    Attempt 2: Gemini 1.5 Flash
    """
    def __init__(self, api_key):
        self.api_key = api_key
        if self.api_key:
            genai.configure(api_key=self.api_key)

        self.model_pro = genai.GenerativeModel("gemini-1.5-pro")
        self.model_flash = genai.GenerativeModel("gemini-1.5-flash")

        # State tracking for UI
        self.current_model_name = "gemini-1.5-pro"
        self.is_fallback_active = False

        print("🧠 Gemini Manager Online: Auto-Switching Enabled.")

    def generate_content(self, prompt):
        if not self.api_key:
            raise Exception("Google API Key not found.")

        # If we already hit the limit previously, stick to Flash to save time
        # (Optional: remove this if if you want to try Pro every single time)
        if self.is_fallback_active:
            return self.model_flash.generate_content(prompt)

        try:
            # 1. Try Pro
            self.current_model_name = "gemini-1.5-pro"
            return self.model_pro.generate_content(prompt)

        except google_exceptions.ResourceExhausted:
            # 2. Catch Quota Error -> Switch to Flash
            print("⚠️ Pro Quota Exceeded. Switching to Flash automatically.")
            self.is_fallback_active = True
            self.current_model_name = "gemini-1.5-flash"
            return self.model_flash.generate_content(prompt)

        except Exception as e:
            # Re-raise other errors (auth, internet, etc.)
            raise e

    def get_status_string(self):
        """Returns the current active model state."""
        icon = "🚀" if not self.is_fallback_active else "⚡"
        return f"{icon} Active Model: {self.current_model_name}"

# Initialize Manager
gemini_manager = GeminiManager(GOOGLE_API_KEY) if GOOGLE_API_KEY else None

# --- HUGGING FACE SYNC MANAGER ---
class HFManager:
    def __init__(self):
        self.api = HfApi(token=HF_TOKEN)
        self.lock = Lock()

    def pull_datasets(self):
        print("⬇️ Pulling datasets from Hugging Face...")
        try:
            files = self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
            csv_files = [f for f in files if f.endswith(".csv")]

            if not csv_files:
                print("⚠️ No CSVs found. Seeding initial data...")
                seed_initial_data()
                return

            for file in csv_files:
                hf_hub_download(
                    repo_id=HF_REPO_ID, filename=file, repo_type="dataset",
                    local_dir=DATASET_DIR, token=HF_TOKEN
                )
        except Exception as e:
            print(f"❌ HF Pull Error: {e}")
            seed_initial_data()

    def push_update(self, filepath, commit_msg="Update from IEDI-MAS"):
        filename = os.path.basename(filepath)
        print(f"⬆️ Pushing update: {filename}...")
        try:
            self.api.upload_file(
                path_or_fileobj=filepath, path_in_repo=filename,
                repo_id=HF_REPO_ID, repo_type="dataset", commit_message=commit_msg
            )
            print("✅ Sync Complete!")
        except Exception as e:
            print(f"❌ HF Push Error: {e}")

    def upload_audio_sample(self, audio_path, dialect):
        clean_dialect = dialect.strip()
        filename = os.path.basename(audio_path)
        hf_path = f"audio/{clean_dialect}/{filename}"
        try:
            self.api.upload_file(
                path_or_fileobj=audio_path, path_in_repo=hf_path,
                repo_id=HF_REPO_ID, repo_type="dataset",
                commit_message=f"Add audio sample for {clean_dialect}"
            )
            return hf_path
        except Exception as e:
            print(f"❌ Audio Upload Error: {e}")
            return None

hf_manager = HFManager()

# ---  DATA SEEDING ---
def seed_initial_data():
    initial_data = {
        "Nigerian English": [{
            "Utterance": "How far?",
            "Clarification": "How are you doing?",
            "Tone_Category": "Casual/Greeting",
            "Linguistic_Context": "Common pidgin greeting functioning like 'What's up?'",
            "Syntax_Pattern": r"\bhow\s?far\b",
            "file_name": ""
        }]
    }
    for dialect, rows in initial_data.items():
        filepath = os.path.join(DATASET_DIR, f"{dialect}.csv")
        if not os.path.exists(filepath):
            df = pd.DataFrame(rows)
            df["Dialect"] = dialect
            df.to_csv(filepath, index=False)
            hf_manager.push_update(filepath, "Initial Seed")

hf_manager.pull_datasets()

# --- AGENT 1: INPUT (Whisper) ---
class AgentInput:
    def __init__(self, model_size="small"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"👂 Agent 1 (Input) Online: Loading Whisper ({model_size}) on {device}...")
        self.model = whisper.load_model(model_size, device=device)

    def transcribe(self, audio_path, language="en"):
        if not audio_path: return []
        result = self.model.transcribe(audio_path, language=language)
        return [{"speaker": "Speaker", "text": seg["text"].strip(), "start": seg["start"], "end": seg["end"]} for seg in result["segments"]]

# --- AGENT 2: INTERPRETATION (Manager-Aware) ---
class AgentInterpretation:
    def __init__(self, gemini_manager_instance=None):
        self.df = pd.DataFrame()
        self.lookup_list = []
        self.gemini_manager = gemini_manager_instance
        self.lab_profile = self.load_lab_profile()
        print("🧠 Agent 2 (Interpretation) Online: Loading Datasets & Lab Context...")
        self.refresh_knowledge_base()

    def load_lab_profile(self):
        profile_path = "nsl_lab_profile.json"
        default_profile = {"lab_name": "General Context", "jargon": {}, "pragmatic_rules": []}
        if os.path.exists(profile_path):
            try:
                with open(profile_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except: return default_profile
        return default_profile

    def save_lab_profile(self, json_str):
        try:
            new_profile = json.loads(json_str)
            with open("nsl_lab_profile.json", "w", encoding="utf-8") as f:
                json.dump(new_profile, f, indent=2)
            self.lab_profile = new_profile
            self.refresh_knowledge_base()
            return "✅ Profile updated! Jargon added to detection logic."
        except: return "❌ Invalid JSON."

    def get_profile_text(self):
        return json.dumps(self.lab_profile, indent=2)

    def refresh_knowledge_base(self):
        all_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
        df_list = []
        for filename in all_files:
            try:
                dialect_name = os.path.basename(filename).replace(".csv", "")
                temp_df = pd.read_csv(filename)
                temp_df["Dialect"] = dialect_name
                df_list.append(temp_df)
            except Exception as e:
                print(f"⚠️ Error loading {filename}: {e}")

        if df_list:
            self.df = pd.concat(df_list, ignore_index=True)
            self.lookup_list = self.df["Utterance"].tolist()
        else:
            self.lookup_list = []

        if self.lab_profile and "jargon" in self.lab_profile:
            jargon_keys = list(self.lab_profile["jargon"].keys())
            self.lookup_list.extend(jargon_keys)

    def detect_dialect(self, text, threshold=75):
        if not self.lookup_list or not text: return "Unknown", None, None, None, None
        match = process.extractOne(text, self.lookup_list, scorer=fuzz.ratio)

        if match:
            best_utterance, score, index = match
            if score >= threshold:
                if index < len(self.df):
                    row = self.df.iloc[index]
                    tone = row.get("Tone_Category", "---")
                    context = row.get("Linguistic_Context", "---")
                    return row["Dialect"], row["Clarification"], best_utterance, tone, context
                else:
                    jargon_def = self.lab_profile["jargon"].get(best_utterance, "Defined in Codebook")
                    profile_dialect = self.lab_profile.get("lab_name", "Custom Profile")
                    return profile_dialect, jargon_def, best_utterance, "Contextual", "Found in Active Persona Codebook"
        return "Unknown", None, None, None, None

    def analyze_pragmatics(self, text, detected_dialect="Unknown", context_window=[]):
        if not self.gemini_manager: return "LLM Offline", "N/A"

        jargon_list = json.dumps(self.lab_profile.get("jargon", {}), indent=2)
        prompt = f"""
        Role: Sociolinguistic Interpreter.
        Active Persona/Jargon: {jargon_list}
        Task: Analyze Tone/Intent. Utterance: "{text}" ({detected_dialect})
        Output: [Tone]: <tone> | [Intent]: <intent>
        """
        try:
            response = self.gemini_manager.generate_content(prompt)
            return response.text.strip()
        except: return "Analysis Failed", "Error"

    def get_rich_suggestions(self, text, dialect):
        if not self.gemini_manager or not text or not dialect:
            return []

        # Logic is inside generate_content now
        profile_context = json.dumps(self.lab_profile, indent=2)
        prompt = f"""
        interpret this {dialect} sentence to standard English: "{text}"

        [CRITICAL CONTEXT]
        Use this specific Cultural Profile/Codebook to guide your interpretation:
        {profile_context}
        If the phrase exists in the 'jargon' above, prioritize that definition.

        [TASK]
        Provide 3 distinct interpretations that capture the most likely Pragmatic Intentions of the speaker.
        Do NOT limit yourself to specific categories like 'Casual' or 'Frustrated'.
        Instead, define the "Tone Category" dynamically based on what fits the phrase best.

        STRICT JSON OUTPUT STRUCTURE:
        [
            {{
                "clarification": "Standard English translation 1",
                "tone": "Dynamic Tone Name",
                "context": "Explanation of cultural usage..."
            }},
            {{
                "clarification": "Translation 2",
                "tone": "Dynamic Tone Name",
                "context": "Explanation..."
            }},
            {{
                "clarification": "Translation 3",
                "tone": "Dynamic Tone Name",
                "context": "Explanation..."
            }}
        ]
        """
        try:
            response = self.gemini_manager.generate_content(prompt)
            clean_text = response.text.strip()
            clean_text = re.sub(r"^```json", "", clean_text)
            clean_text = re.sub(r"^```", "", clean_text)
            clean_text = re.sub(r"```$", "", clean_text)
            clean_text = clean_text.strip()
            data = json.loads(clean_text)
            return data
        except Exception as e:
            print(f"❌ Rich Suggestion Error: {e}")
            return []

    def generate_syntax_pattern(self, utterance):
        if not self.gemini_manager: return r"\b" + utterance.lower().replace(" ", r"\s?") + r"\b"
        prompt = f"Create Python Regex for: '{utterance}'. Output ONLY regex."
        try:
            response = self.gemini_manager.generate_content(prompt)
            return response.text.strip()
        except: return r"\b" + utterance.lower().replace(" ", r"\s?") + r"\b"

# --- AGENT 4: TRUST (IPFS & CSV Persistence) ---
class AgentTrust:
    def __init__(self):
        self.lock = Lock()
        print("🛡️ Agent 4 (Trust) Online.")

    def log_to_ipfs(self, data):
        if not PINATA_JWT: return "Local-Log-Only"
        headers = {"Authorization": f"Bearer {PINATA_JWT}"}
        try:
            res = requests.post("https://api.pinata.cloud/pinning/pinJSONToIPFS", headers=headers, json=data)
            return res.json().get("IpfsHash", "Error")
        except: return "IPFS_Fail"

    def process_feedback(self, action, original_text, dialect, clarification, tone, context, brain_agent, audio_path=None):
        timestamp = pd.Timestamp.now().isoformat()
        feedback_data = {
            "original": original_text, "dialect": dialect, "clarification": clarification,
            "tone": tone, "linguistic_context": context, "action": action, "timestamp": timestamp
        }
        self.log_to_ipfs(feedback_data)

        if action == "Suggest Update":
            syntax = brain_agent.generate_syntax_pattern(original_text)
            update_msg = self.update_dataset_csv(dialect, original_text, clarification, tone, context, syntax, audio_path)
            brain_agent.refresh_knowledge_base()
            return f"{update_msg}\n🤖 Syntax: {syntax}"
        return "Feedback Logged."

    def update_dataset_csv(self, dialect, utterance, clarification, tone, context, syntax, audio_path=None):
        clean_dialect = dialect.strip().title()
        if not clean_dialect.endswith("English") and not clean_dialect.endswith("Dialect"):
             clean_dialect += " Dialect"
        filepath = os.path.join(DATASET_DIR, f"{clean_dialect}.csv")

        with self.lock:
            if not os.path.exists(filepath):
                new_df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "file_name"])
                new_df.to_csv(filepath, index=False)

            df = pd.read_csv(filepath)
            for col in ["Tone_Category", "Linguistic_Context", "file_name", "Syntax_Pattern"]:
                if col not in df.columns: df[col] = "---"

            audio_ref = ""
            if audio_path and os.path.exists(audio_path):
                # 1. GENERATE UNIQUE NAME
                ext = os.path.splitext(audio_path)[1]
                # We add a random ID so multiple submissions of the same audio don't overwrite each other in the dataset folder
                unique_name = f"{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{random.randint(1000,9999)}{ext}"
                new_path = os.path.join(os.path.dirname(audio_path), unique_name)

                # 2. FIX: COPY INSTEAD OF MOVE (shutil.copy2)
                # This keeps the original 'audio_path' valid for the next submission
                try:
                    shutil.copy2(audio_path, new_path)
                    audio_ref = hf_manager.upload_audio_sample(new_path, dialect)
                except Exception as e:
                    print(f"❌ Error copying audio: {e}")
                    audio_ref = "Error_Saving_Audio"

            # SMART UPDATE LOGIC
            exact_match = df[(df["Utterance"] == utterance) & (df["Clarification"] == clarification)]

            if not exact_match.empty:
                idx = exact_match.index[0]
                df.loc[idx, "Syntax_Pattern"] = syntax
                df.loc[idx, "Tone_Category"] = tone
                df.loc[idx, "Linguistic_Context"] = context
                if audio_ref: df.loc[idx, "file_name"] = audio_ref
                msg = f"🔄 Updated metadata for existing: '{utterance}'"
            else:
                new_row = pd.DataFrame([{
                    "Utterance": utterance, "Dialect": clean_dialect, "Clarification": clarification,
                    "Tone_Category": tone, "Linguistic_Context": context,
                    "Syntax_Pattern": syntax, "file_name": audio_ref
                }])
                df = pd.concat([df, new_row], ignore_index=True)
                msg = f"✅ Added NEW definition for: '{utterance}'"

            df.to_csv(filepath, index=False)
            hf_manager.push_update(filepath, f"Update {clean_dialect}: {utterance}")
            return msg

# --- AGENT 3: UX (Trigger-Responsive Interface) ---
class AgentUX:
    def __init__(self, input_agent, brain_agent, trust_agent):
        self.input = input_agent
        self.brain = brain_agent
        self.trust = trust_agent
        self.last_audio_path = None
        self.suggestion_cache = {}
        print("🎨 Agent 3 (UX) Online: Building Interface...")

    def get_quota_status(self):
        """Fetches current usage from the Gemini Manager."""
        if self.brain.gemini_manager:
            return self.brain.gemini_manager.get_status_string()
        return "Manager not active"

    def automated_pipeline(self, audio_path, language="en"):
        if not audio_path:
            return pd.DataFrame(), "Waiting for Input...", self.get_quota_status()

        self.last_audio_path = audio_path
        segments = self.input.transcribe(audio_path, language)
        results = []
        history = []
        for seg in segments:
            raw = seg["text"]
            d, c, m, tone, ctx = self.brain.detect_dialect(raw)
            disp = m if m else raw
            t_disp = tone if tone else "---"
            c_disp = ctx if ctx else "---"
            prag = self.brain.analyze_pragmatics(disp, d, history) if d != "Unknown" else "---"
            results.append({
                "Speaker": seg["speaker"],
                "Utterance": disp,
                "Dialect": d,
                "Clarification": c if c else "---",
                "Tone": t_disp,
                "Context": c_disp,
                "Pragmatic Analysis": prag
            })
            history.append(disp)

        return pd.DataFrame(results), "✅ Analysis Complete", self.get_quota_status()

    def launch(self):
        existing_dialects = []
        if os.path.exists(DATASET_DIR):
            csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
            existing_dialects = [os.path.basename(f).replace(".csv", "") for f in csv_files]
        dropdown_choices = existing_dialects + ["+ Add New Dialect"]

        with gr.Blocks(theme=gr.themes.Soft()) as ui:
            gr.Markdown("## 🌍 IEDI-M²: Active Listening & Dialect Mediator")

            with gr.Tabs():
                with gr.Tab("🎙️ Live Analysis"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            audio_input = gr.Audio(label="Step 1: Speak/Upload", sources=["microphone", "upload"], type="filepath")
                            lang_select = gr.Dropdown(["en", "ko", "fr"], value="en", label="Step 2: Language (Optional)")
                            analyze_btn = gr.Button("Re-Run Analysis 🔄", variant="secondary")

                            quota_display = gr.Textbox(
                                label="📊 Model Status",
                                value=self.get_quota_status(),
                                interactive=False
                            )

                        with gr.Column(scale=2):
                            status_box = gr.Textbox(label="Status", interactive=False)
                            results_df = gr.Dataframe(
                                headers=["Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"],
                                interactive=False,
                                label="Analysis Results",
                                type="pandas",
                                wrap=True
                            )

                    gr.Markdown("### ✍️ Active Feedback Loop (Agent 4: Trust)")
                    with gr.Row():
                        with gr.Column(scale=1):
                            orig_text_state = gr.Textbox(visible=True, label="Original Text (Edit to Correct)")
                            with gr.Row():
                                dialect_dropdown = gr.Dropdown(choices=dropdown_choices, label="Select Dialect", interactive=True)
                                new_dialect_input = gr.Textbox(label="Enter New Dialect Name", visible=False, interactive=True)

                        with gr.Column(scale=1):
                            suggestion_dropdown = gr.Dropdown(
                                label="Suggest Clarification (Select Option)",
                                choices=[],
                                allow_custom_value=True,
                                interactive=True
                            )
                            selected_tone_state = gr.Textbox(label="Linguistic Tone", interactive=True)
                            selected_context_state = gr.TextArea(label="Linguistic Context", interactive=True, lines=2)

                    with gr.Row():
                        btn_accept = gr.Button("✅ Accept", variant="secondary")
                        btn_reject = gr.Button("❌ Reject", variant="stop")
                        btn_suggest = gr.Button("💾 Suggest Update", variant="primary")

                    feedback_out = gr.Markdown()

                with gr.Tab("⚙️ Lab Context"):
                    gr.Markdown("### 📝 Edit the 'Lab Persona' Codebook")
                    profile_editor = gr.Code(value=self.brain.get_profile_text, language="json", label="nsl_lab_profile.json", lines=20)
                    save_profile_btn = gr.Button("💾 Save & Reload Profile", variant="primary")
                    profile_status = gr.Textbox(label="System Response", interactive=False)

            # --- EVENT LOGIC ---
            def update_suggestions_rich(text, dialect):
                try:
                    if not text or not dialect or dialect == "+ Add New Dialect":
                        return gr.update(choices=[]), "", "", self.get_quota_status()

                    print(f"⏳ Generating rich suggestions for '{text}' in {dialect}...")
                    suggestions_data = self.brain.get_rich_suggestions(text, dialect)

                    self.suggestion_cache = {}
                    display_choices = []

                    if not suggestions_data:
                         return gr.update(choices=["No suggestions available"]), "", "", self.get_quota_status()

                    for item in suggestions_data:
                        clar = item.get("clarification", "")
                        tone = item.get("tone", "General")
                        ctx = item.get("context", "")
                        display_str = f"{clar}  [{tone}]"
                        display_choices.append(display_str)
                        self.suggestion_cache[display_str] = {"clar": clar, "tone": tone, "context": ctx}

                    if display_choices:
                        first_key = display_choices[0]
                        first_data = self.suggestion_cache[first_key]
                        return gr.update(choices=display_choices, value=first_key), first_data["tone"], first_data["context"], self.get_quota_status()
                    else:
                        return gr.update(choices=[]), "", "", self.get_quota_status()
                except Exception as e:
                    return gr.update(choices=["Error"]), str(e), "", self.get_quota_status()

            def on_suggestion_select(val):
                if val in self.suggestion_cache:
                    data = self.suggestion_cache[val]
                    return data["tone"], data["context"]
                return "Custom/User Edited", "User provided context"

            # Wiring Updates
            dialect_dropdown.change(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, quota_display])
            orig_text_state.blur(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, quota_display])
            suggestion_dropdown.change(fn=on_suggestion_select, inputs=[suggestion_dropdown], outputs=[selected_tone_state, selected_context_state])

            # --- PIPELINE WIRING (SINGLE TRIGGER FIX) ---
            audio_input.change(self.automated_pipeline, [audio_input, lang_select], [results_df, status_box, quota_display])
            analyze_btn.click(self.automated_pipeline, [audio_input, lang_select], [results_df, status_box, quota_display])

            def handle_selection(evt: gr.SelectData, df):
                if df is None or len(df) == 0: return "", "", "", "", ""
                try:
                    row = df.iloc[evt.index[0]]
                    d = row["Dialect"] if row["Dialect"] in existing_dialects else None
                    return row["Utterance"], d, row["Clarification"], row["Tone"], row["Context"]
                except: return "", "", "", "", ""

            results_df.select(handle_selection, [results_df], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state])

            def submit_logic(action, orig, d_drop, d_new, clar_raw, tone, context):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    if not final_d or not orig: return "❌ Invalid Input"
                    final_clar = clar_raw
                    if "[" in str(clar_raw) and "]" in str(clar_raw):
                         final_clar = str(clar_raw).rsplit("[", 1)[0].strip()
                    audio_ref = self.last_audio_path if action == "Suggest Update" else None
                    return self.trust.process_feedback(action, orig, final_d, final_clar, tone, context, self.brain, audio_ref)
                except Exception as e: return f"❌ Error submitting: {e}"

            btn_suggest.click(lambda o, d, n, c, t, ctx: submit_logic("Suggest Update", o, d, n, c, t, ctx), [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state], [feedback_out])

            def on_dialect_change(val): return gr.update(visible=True) if val == "+ Add New Dialect" else gr.update(visible=False)
            dialect_dropdown.change(on_dialect_change, inputs=dialect_dropdown, outputs=new_dialect_input)
            save_profile_btn.click(self.brain.save_lab_profile, inputs=[profile_editor], outputs=[profile_status])

        ui.launch(share=True, debug=True)

# --- START SYSTEM ---
agent1 = AgentInput()
agent2 = AgentInterpretation(gemini_manager) # Pass the Dynamic Manager
agent4 = AgentTrust()
agent3 = AgentUX(agent1, agent2, agent4)
agent3.launch()

🧠 Gemini Manager Online: Auto-Switching Enabled.
⬇️ Pulling datasets from Hugging Face...
👂 Agent 1 (Input) Online: Loading Whisper (small) on cuda...
🧠 Agent 2 (Interpretation) Online: Loading Datasets & Lab Context...
🛡️ Agent 4 (Trust) Online.
🎨 Agent 3 (UX) Online: Building Interface...


/tmp/ipython-input-1942893554.py:481: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as ui:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://049d32c23efb965cf7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://049d32c23efb965cf7.gradio.live
